In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Callable
from dataclasses import dataclass

# Activation function class
class TanhAct:
    def forward(self, x):
        self.output = np.tanh(x)
        return self.output
    
    def backward(self, d_out):
        return (1 - self.output ** 2) * d_out

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)



In [ ]:
# RNN Model for Many-to-One task
class VanillaRNN:
    def __init__(self, input_size: int, n_neurons: int, output_size: int, Act=TanhAct):
        self.input_size = input_size
        self.n_neurons = n_neurons
        self.output_size = output_size

        self.Wx = np.random.randn(n_neurons, input_size) * np.sqrt(1. / input_size)
        self.Wh = np.random.randn(n_neurons, n_neurons) * np.sqrt(1. / n_neurons)
        self.Wy = np.random.randn(output_size, n_neurons) * np.sqrt(1. / n_neurons)
        self.bh = np.zeros((n_neurons, 1))
        self.by = np.zeros((output_size, 1))

        self.Act = Act

    def forward(self, inputs: List[np.ndarray]) -> np.ndarray:
        T = len(inputs)
        self.H = [np.zeros((self.n_neurons, 1)) for _ in range(T + 1)]
        self.activations = [self.Act() for _ in range(T)]

        for t in range(T):
            x_t = inputs[t]
            self.H[t + 1] = self.activations[t].forward(np.dot(self.Wx, x_t) + np.dot(self.Wh, self.H[t]) + self.bh)
        
        y_hat = np.dot(self.Wy, self.H[T]) + self.by
        return y_hat, self.H[T]

    def backward(self, d_y, H, inputs: List[np.ndarray]):
        T = len(inputs)
        dWx = np.zeros_like(self.Wx)
        dWh = np.zeros_like(self.Wh)
        dWy = np.zeros_like(self.Wy)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)

        dht_next = np.zeros((self.n_neurons, 1))

        for t in reversed(range(T)):
            dtanh = self.activations[t].backward(dht_next)

            dWy += np.dot(d_y, H[t + 1].T)
            dby += d_y

            dht = np.dot(self.Wy.T, d_y) + dht_next

            dWx += np.dot(dtanh, inputs[t].T)
            dWh += np.dot(dtanh, self.H[t].T)
            dbh += dtanh

            dht_next = np.dot(self.Wh.T, dtanh)

        return dWx, dWh, dWy, dbh, dby

    def train(self, data, epochs: int, learning_rate: float):
        for epoch in range(epochs):
            total_loss = 0
            num_correct = 0

            for x, y in data:
                inputs = word2vec(x)
                target = int(y)

                # Forward pass
                out, last_hidden_state = self.forward(inputs)
                probs = softmax(out)

                # Loss and accuracy